In [ ]:
import pandas as pd

try:
    df_pop = pd.read_excel('Население по регионам 2024.xlsx')
except FileNotFoundError:
    print("Ошибка: Файл 'Население по регионам 2024.xlsx' не найден.")
    exit()

try:
    df_circus = pd.read_excel('Общий свод по циркам 2024.xlsx', header=4)
except FileNotFoundError:
    print("Ошибка: Файл со статистикой цирков не найден.")
    exit()

df_pop.columns = [str(col).strip() for col in df_pop.columns]
df_circus.columns = [str(col).strip() for col in df_circus.columns]

cols_to_keep = ['А', '3', '7', '34', '36', '39', '121', '19', '25']

missing = [c for c in cols_to_keep if c not in df_circus.columns]
if missing:
    print(f"Ошибка: В таблице цирков не найдены колонки: {missing}")
    print("Доступные колонки:", df_circus.columns.tolist())
    exit()

df_circus_filtered = df_circus[cols_to_keep].copy()

df_circus_filtered.rename(columns={
    'А': 'Регион',
    '3': 'Количество стационарных цирков, ед.',
    '7': 'Количество цирков шапито, ед.',
    '34': 'Количество цирковых представлений, ед.',
    '36': 'Количество зрителей на цирковых представлениях, тыс. чел.',
    '39': 'Количество проданных билетов, тыс. шт.',
    '121': 'Полученные средства от цирковых представлений, тыс. руб.',
    '19': 'Артистический персонал, чел.',
    '25': 'Количество животных, ед.'
}, inplace=True)

df_final = pd.merge(
    df_pop,
    df_circus_filtered,
    left_on='Регион (очищенный)',
    right_on='Регион',
    how='inner'
)

if 'Регион (очищенный)' in df_final.columns:
    df_final.drop(columns=['Регион (очищенный)'], inplace=True)

df_final.rename(columns={'Население, чел': 'Население, чел.'}, inplace=True)

all_cols = df_final.columns.tolist()
main_cols = ['Регион', 'Население, чел.']
other_cols = [c for c in all_cols if c not in main_cols]
df_final = df_final[main_cols + other_cols]

print("Объединение завершено успешно!")
print(f"Итого строк: {len(df_final)}")
print(df_final.head())

df_final.to_excel('Результат_Объединения_2024.xlsx', index=False)
print("Файл сохранен как 'Результат_Объединения_2024.xlsx'")